## Rate Limiting and Exponential Backoff

### 1. What is rate limiting?

Rate limiting controls how many requests a client is allowed to make within a period of time.

For example, suppose an API says:

60 requests per minute

You cannot send 100 requests immediately. The API may reject requests after the limit is reached.

A simple example:

```
Allowed: 60 requests / minute

Request 1   ✓
Request 2   ✓
...
Request 60  ✓
Request 61  ✗  → HTTP 429 Too Many Requests
```

The purpose is to:

- Protect the server
- Prevent abuse
- Ensure fair usage
- Prevent overload
- Control infrastructure costs

### 2. Rate limiting vs exponential backoff

Think of them as two different mechanisms.

- Rate limiting = "How fast am I allowed to send requests?"
- Exponential backoff = "What should I do after a request fails?"

For example:

```
Client
  |
  | Request
  v
API
  |
  | 429 Too Many Requests
  v
Client
  |
  | wait 1 second
  | retry
  |
  | 429
  |
  | wait 2 seconds
  | retry
  |
  | 429
  |
  | wait 4 seconds
  | retry
```

So:

Rate limiter controls request frequency.

Backoff controls retry timing.

### 3. Why exponential backoff?

Suppose your application gets a 429.

A bad implementation would be:

```
while True:
    response = call_api()

    if response.status_code == 429:
        continue
```

This creates a retry storm:

```
Request → 429
Request → 429
Request → 429
Request → 429
Request → 429
...
```

You're actually making the server's problem worse.

Instead, use exponential backoff:

```
Attempt 1 → wait 1 sec
Attempt 2 → wait 2 sec
Attempt 3 → wait 4 sec
Attempt 4 → wait 8 sec
Attempt 5 → wait 16 sec
```

The basic formula is:
```
delay = base_delay^retry_attempt
```
For example:
```
base = 1 second


attempt 0 → 1 second
attempt 1 → 2 seconds
attempt 2 → 4 seconds
attempt 3 → 8 seconds
attempt 4 → 16 seconds
```

Usually you also add jitter:
```
delay = exponential_backoff + random_jitter
```
because otherwise thousands of clients might retry at exactly the same time.

### 4. How do you implement it?

Let's say an API allows:

5 requests/second

and occasionally returns 429.

A simple retry implementation could look like:

In [3]:
import time
import random
import requests

BASE_DELAY = 2

def fetch(url, max_retries=5):

    for attempt in range(max_retries):

        response = requests.get(url)

        if response.status_code == 200:
            return response

        if response.status_code == 429:
            delay = (BASE_DELAY ** attempt) + random.uniform(0, 1)  # adds random jitter

            print(f"Rate limited. Waiting {delay} seconds...")
            time.sleep(delay)

            continue

        # Other errors
        response.raise_for_status()

    raise Exception("Max retries exceeded")

### 5. Now let's talk specifically about LLM APIs

The concept is the same.

For example, you call an LLM API:
```
response = client.chat.completions.create(...)
```
and the provider responds:

429 Too Many Requests

You should generally:
```
429
 ↓
wait
 ↓
retry
```
with exponential backoff.

But LLM APIs introduce an important complication:

There can be multiple kinds of rate limits.

For example:
```
Requests Per Minute (RPM)
Tokens Per Minute (TPM)
Requests Per Day
Daily token quota
```

So you might make only:

10 requests

but each request contains:

20,000 tokens

and exceed your token limit.

That's why LLM rate limiting isn't necessarily just:

"number of requests per second."

### 6. Example with an LLM

Imagine an LLM provider gives you:

60 requests/minute
100,000 tokens/minute

You send:
```
Request 1 → 5,000 tokens
Request 2 → 5,000 tokens
...
```
After enough requests, you could hit either:

RPM limit

or:

TPM limit

The provider may respond with:

429

At that point, exponential backoff is appropriate.

### 7. Very important: Retry-After

Some APIs tell you exactly how long to wait.

For example:
```
HTTP 429
Retry-After: 7
```

This means:

Wait 7 seconds before trying again.

In that situation, you should generally respect that instruction rather than blindly calculating:
```
2 ** attempt
```
Conceptually:
```
if response.status_code == 429:


    if "Retry-After" in response.headers:
        delay = int(response.headers["Retry-After"])
    else:
        delay = 2 ** attempt


    time.sleep(delay)
```
So your strategy becomes:
```
Did server tell me when to retry?
          |
       YES → use Retry-After
          |
       NO
          ↓
Use exponential backoff
```

### 8. How would you handle rate limiting when calling an LLM API?

A strong answer would be:

*"I would implement client-side rate limiting based on the provider's RPM and, where applicable, token-per-minute limits, so that we don't continuously exceed the service quota. If the provider still returns a 429, I would retry using exponential backoff, preferably respecting the Retry-After header if provided. I'd also add jitter to prevent synchronized retries when multiple workers receive rate-limit responses simultaneously. I'd cap the maximum backoff and number of retries, and only retry errors that are actually transient."*

### 9. Does that mean the server side LLM doesn't do the exponential backoff on its side and just returns exception with rate limiting message?


Generally, yes, but there's an important distinction.

When you call an LLM API and you're over the provider's rate limit, the server typically doesn't sit there doing exponential backoff for your request.

Instead, it usually does something like:

```
Your application
      |
      |  Request
      ↓
LLM API server
      |
      | "You're over the limit"
      ↓
   HTTP 429
      |
      ↓
Your application
```

Your application is responsible for deciding:

```
429 received
     ↓
wait
     ↓
retry
```

Think of it this way

- The server-side rate limiter says:

*"No. You're sending too many requests right now."*

- The client-side exponential backoff says:

*"Okay, I'll wait progressively longer before asking again."*

### One subtle point

The LLM provider may itself have internal retry/backoff mechanisms between its own services.

For example:

```
Your app
   ↓
LLM API Gateway
   ↓
Internal service
   ↓
GPU/model server
```

The provider can use retries/backoff *internally* between those components.

But that's different from your API request.

From your application's perspective, if the API returns:
```
429 Too Many Requests
```
you should assume you need to handle the retry/backoff behavior on the client side, unless the SDK explicitly documents that it automatically retries.